# Sesión 8 · Estación 2 · Del criterio cualitativo a la codificación con IA

**El caso.** Una investigadora estudiaba las llamadas a la línea de emergencias 123. No programaba. Desde su experticia cualitativa leyó cientos de minutas y creó categorías de **repertorios de violencia**: las formas reconocibles que toma un conflicto (en la casa o en la calle, con golpes o con palabras, para quitar algo o para intimidar). Después, un científico de datos codificó miles de llamadas con esas categorías. Hoy la IA puede hacer la parte del científico de datos. **La categoría sigue naciendo del criterio cualitativo.**

**El marco.** Nelson (2020) lo llama *teoría fundamentada computacional*: el computador detecta y confirma patrones a escala, y la lectura experta los refina. Nelson, L. K. (2020). Computational Grounded Theory: A Methodological Framework. *Sociological Methods & Research*, 49(1), 3–42. https://doi.org/10.1177/0049124117729703

**Los datos.** 80 llamadas **simuladas** de «Ciudad del Valle», una ciudad inventada (octubre de 2026). Una experta ya codificó 60 de ellas; con esas se mide su acuerdo. No son datos reales.

**Lo que va a hacer:**
1. Leer llamadas antes de clasificar.
2. Escribir **su** libro de códigos.
3. Pedirle a la IA de Colab que codifique las 80 llamadas con su libro.
4. Armar la tabla de equivalencias entre sus categorías y las de la experta.
5. Medir el acuerdo y leer los desacuerdos.
6. Pedirle a la IA los cinco casos que peor encajan.

In [ ]:
import pandas as pd

BASE = "https://alonsorh-econ.github.io/diplomado-ia-investigacion/sesion-08/datos/"
llamadas = pd.read_csv(BASE + "llamadas_estudiantes.csv")
experta = pd.read_csv(BASE + "muestra_experta.csv")
print(f"{len(llamadas)} llamadas para codificar · {len(experta)} ya codificadas por la experta (no las mire todavía)\n")

# Las primeras 12, para leer antes de clasificar
for f in llamadas.head(12).itertuples():
    print(f"{f.id_llamada} · {f.dia_semana} {f.fecha} {f.hora} · {f.barrio}")
    print(f"   {f.resumen}\n")

## Paso 1 · Lea antes de clasificar

Lea las 12 llamadas de arriba. Anote qué formas de violencia aparecen, qué las distingue entre sí y cuáles le cuesta clasificar. **No abra todavía la muestra de la experta**: el ejercicio es que las categorías salgan de su lectura.

## Paso 2 · Escriba su libro de códigos

Escríbalo en la celda de abajo, en texto libre. Para cada categoría: **nombre, definición, incluye, excluye y un ejemplo**. Agregue una categoría **OTRO** y una **regla de desempate**: qué hacer cuando una llamada cabe en dos categorías.

La celda empieza con `%%writefile`: esa línea guarda su texto en un archivo para que la IA lo pueda leer. **No la borre.** Borre el ejemplo, escriba el suyo y corra la celda.

*Consejos:* entre cuatro y ocho categorías funciona bien. Escriba los nombres en MAYÚSCULAS: la IA los va a copiar tal cual, y así es más fácil compararlos después.

In [ ]:
%%writefile mi_libro_de_codigos.md
MI LIBRO DE CÓDIGOS (esto es un ejemplo corto: bórrelo y escriba el suyo)

CATEGORÍA: PELEA EN LA CASA
Definición: agresión entre personas que son pareja o familia.
Incluye: golpes, empujones o insultos entre pareja o familiares.
Excluye: peleas entre desconocidos en la calle.
Ejemplo: «Mujer reporta que su esposo la empujó en la cocina.»

CATEGORÍA: PELEA EN LA CALLE
Definición: agresión física entre personas sin vínculo, en un lugar público.
Incluye: riñas a la salida de bares, peleas después de un partido.
Excluye: peleas de pareja, aunque ocurran en la calle.
Ejemplo: «Vigilante reporta que dos hombres se golpean frente a una tienda.»

CATEGORÍA: OTRO
Definición: llamadas que no caben en ninguna categoría.

REGLA DE DESEMPATE: si una llamada cabe en dos categorías, gana la que describe el hecho más grave.

## Paso 3 · La IA codifica con su libro

La celda de abajo le pasa a la IA de Colab su libro de códigos y **una llamada a la vez**, y le pide una sola línea con este formato:

`CATEGORIA | frase que la justifica | encaja: si/no`

Tarda unos 8 segundos por llamada: las 80 toman alrededor de 11 minutos. Para probar primero, cambie `N_MAX = 80` por `N_MAX = 10` y vuelva a 80 cuando vea que funciona. El resultado queda en `codificadas_por_mi.csv`. **Mientras corre, lea las justificaciones.**

Si su Colab no tiene la IA, la celda le avisa y usted sigue con el plan B (dos celdas más abajo).

In [ ]:
import re
import time

N_MAX = 80          # baje a 10 para una prueba rápida
CONTINUAR = False   # póngalo en True si se cortó a mitad de camino y quiere seguir donde iba

PLANTILLA = """Eres un asistente de investigación cualitativa. Clasifica UNA llamada a la línea de emergencias 123 usando SOLO el libro de códigos de abajo.

LIBRO DE CÓDIGOS:
{libro}

LLAMADA:
{resumen}

Responde en UNA sola línea, sin nada más, con este formato exacto:
CATEGORIA | frase de la llamada que justifica la categoría | encaja: si/no
Usa el nombre de la categoría tal como aparece en el libro. Escribe «encaja: no» si la llamada no cabe bien en ninguna categoría o si cabe en dos."""


def leer_respuesta(texto):
    """Toma la respuesta de la IA y saca categoría, justificación y si encaja (con tolerancia)."""
    texto = (texto or "").strip()
    lineas = [l for l in texto.splitlines() if l.strip()]
    linea = next((l for l in lineas if "|" in l), lineas[0] if lineas else "")
    partes = [p.strip(" *`\"'\t") for p in linea.split("|")]
    categoria = re.sub(r"^(\d+[.)-]\s*|categor[ií]a\s*:\s*)", "", partes[0], flags=re.I).strip() if partes else ""
    justificacion = partes[1] if len(partes) > 1 else ""
    m = re.search(r"encaja\s*:?\s*(s[ií]|no)", texto, flags=re.I)
    encaja = ("si" if m.group(1).lower().startswith("s") else "no") if m else ""
    return categoria.upper(), justificacion, encaja


try:
    from google.colab import ai
except ImportError:
    ai = None
    print("Si su Colab no tiene `google.colab.ai`, use el plan B: suba el CSV a gemini.google.com "
          "con su libro de códigos y pegue el resultado aquí.")
    print("(El plan B está dos celdas más abajo.)")

libro = None
try:
    with open("mi_libro_de_codigos.md", encoding="utf-8") as fh:
        libro = fh.read()
except FileNotFoundError:
    print("Primero corra la celda de su libro de códigos (la que empieza con %%writefile).")

if ai is not None and libro is not None:
    hechas = {}
    if CONTINUAR:
        try:
            previas = pd.read_csv("codificadas_por_mi.csv")
            hechas = {f.id_llamada: f._asdict() for f in previas.itertuples(index=False) if f.categoria != "ERROR"}
            print(f"Sigo donde iba: {len(hechas)} llamadas ya estaban codificadas.")
        except FileNotFoundError:
            pass

    filas, t0 = [], time.time()
    tanda = llamadas.head(N_MAX)
    for k, f in enumerate(tanda.itertuples(), start=1):
        if f.id_llamada in hechas:
            filas.append(hechas[f.id_llamada])
        else:
            respuesta, error = None, ""
            for intento in range(2):          # si falla, reintenta una vez
                try:
                    respuesta = ai.generate_text(PLANTILLA.format(libro=libro, resumen=f.resumen))
                    break
                except Exception as e:
                    error = str(e)[:150]
                    time.sleep(3)
            if respuesta is None:
                filas.append(dict(id_llamada=f.id_llamada, categoria="ERROR", justificacion=error,
                                  encaja="", respuesta=""))
            else:
                categoria, justificacion, encaja = leer_respuesta(respuesta)
                filas.append(dict(id_llamada=f.id_llamada, categoria=categoria or "ERROR",
                                  justificacion=justificacion, encaja=encaja, respuesta=respuesta.strip()))
        if k % 10 == 0 or k == len(tanda):
            pd.DataFrame(filas).to_csv("codificadas_por_mi.csv", index=False)   # guarda por tramos
            print(f"{k} de {len(tanda)} llamadas · {time.time() - t0:.0f} s")

    mias = pd.DataFrame(filas)
    mias.to_csv("codificadas_por_mi.csv", index=False)
    n_error = int((mias.categoria == "ERROR").sum())
    print(f"\nListo: codificadas_por_mi.csv ({len(mias)} llamadas, {n_error} con ERROR).")
    if n_error > len(mias) / 4:
        print("Muchas llamadas salieron con ERROR (quizá el límite de su cuenta gratis): use el plan B.")
    display(mias[["id_llamada", "categoria", "justificacion", "encaja"]].head(10))

### Plan B · Si su Colab no tiene `google.colab.ai`, o si muchas llamadas salieron con ERROR

1. Descargue [llamadas_estudiantes.csv](https://alonsorh-econ.github.io/diplomado-ia-investigacion/sesion-08/datos/llamadas_estudiantes.csv).
2. En [gemini.google.com](https://gemini.google.com) suba el archivo y pegue este prompt, seguido de su libro de códigos:

```text
Eres un asistente de investigación cualitativa. Clasifica cada llamada del archivo adjunto usando SOLO el libro de códigos que te pego abajo. Devuélveme una línea por llamada, sin encabezados ni comentarios, con este formato exacto:
id_llamada | CATEGORIA | frase de la llamada que justifica la categoría | encaja: si/no
Usa el nombre de la categoría tal como aparece en el libro. Escribe «encaja: no» si la llamada no cabe bien en ninguna categoría o si cabe en dos. Son 80 llamadas: no te saltes ninguna.

LIBRO DE CÓDIGOS:
```

3. Copie la respuesta, péguela en la celda de abajo entre las comillas triples y corra la celda.

*Plan C, solo para ver cómo funciona el cálculo:* ponga `USAR_EJEMPLO = True` y use la codificación de ejemplo de una IA que trabajó con el libro de la experta.

In [ ]:
import re

RESULTADO_PEGADO = """

"""
USAR_EJEMPLO = False   # plan C


def leer_linea_pegada(linea):
    m = re.search(r"\bL\d{3}\b", linea)
    if not m:
        return None
    resto = linea[m.end():].strip(" |,;:\t")
    sep = "|" if "|" in resto else ("," if "," in resto else None)
    partes = [p.strip(" *`\"'") for p in (resto.split(sep) if sep else [resto])]
    e = re.search(r"encaja\s*:?\s*(s[ií]|no)", linea, flags=re.I)
    return dict(id_llamada=m.group(0),
                categoria=(partes[0] if partes else "").upper(),
                justificacion=partes[1] if len(partes) > 1 else "",
                encaja=("si" if e.group(1).lower().startswith("s") else "no") if e else "",
                respuesta=linea.strip())


if USAR_EJEMPLO:
    mias = pd.read_csv(BASE + "codificacion_ejemplo_ia.csv").rename(columns={"repertorio": "categoria"})
    mias = mias.assign(justificacion="", encaja="", respuesta="")
    mias.to_csv("codificadas_por_mi.csv", index=False)
    print(f"Plan C: se cargó la codificación de ejemplo ({len(mias)} llamadas).")
elif RESULTADO_PEGADO.strip():
    filas = [x for x in (leer_linea_pegada(l) for l in RESULTADO_PEGADO.splitlines()) if x]
    mias = pd.DataFrame(filas).drop_duplicates("id_llamada")
    mias.to_csv("codificadas_por_mi.csv", index=False)
    faltan = sorted(set(llamadas.id_llamada) - set(mias.id_llamada))
    print(f"Plan B: se leyeron {len(mias)} llamadas y se guardaron en codificadas_por_mi.csv.")
    if faltan:
        print(f"Faltan {len(faltan)}: {', '.join(faltan[:15])}{' …' if len(faltan) > 15 else ''}")
    display(mias.head(10))
else:
    print("No hay nada pegado. Si ya corrió el plan A, siga al paso 4.")

## Paso 4 · La tabla de equivalencias

Usted y la experta usaron libros de códigos distintos, con nombres distintos. **Comparar libros de códigos distintos exige una tabla de equivalencias:** antes de medir el acuerdo hay que decir a qué categoría de la experta corresponde cada una de las suyas.

| Código de la experta | Categoría |
|---|---|
| `vif_pareja` | Violencia intrafamiliar o de pareja (en la casa o en la calle) |
| `rina` | Riña en espacio público entre personas sin vínculo |
| `amenaza` | Amenaza o intimidación sin agresión física |
| `hurto_violento` | Hurto con violencia o intimidación (el intento cuenta) |
| `acoso_mujer` | Acoso o violencia contra la mujer en espacio público |
| `convivencia` | Convivencia sin violencia: ruido, vecinos, animales |
| `otro` | No corresponde a ningún repertorio |

Puede mandar dos categorías suyas al mismo código. Si una categoría suya mezcla dos de la experta, escoja la más cercana: los desacuerdos le van a mostrar el costo de esa decisión.

In [ ]:
import re
import unicodedata

mias = pd.read_csv("codificadas_por_mi.csv").fillna("")
print("Sus categorías y cuántas llamadas recibió cada una:")
print(mias["categoria"].value_counts().to_string(), "\n")

# ---- Escriba aquí su tabla de equivalencias: SU categoría → código de la experta ----
# Ejemplo con el libro de muestra (bórrelo y ponga las suyas):
EQUIVALENCIAS = {
    "PELEA EN LA CASA": "vif_pareja",
    "PELEA EN LA CALLE": "rina",
    "OTRO": "otro",
}
# ------------------------------------------------------------------------------------

CODIGOS_EXPERTA = ["vif_pareja", "rina", "amenaza", "hurto_violento", "acoso_mujer", "convivencia", "otro"]


def normalizar(x):
    x = unicodedata.normalize("NFKD", str(x)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", x).strip(" .:*`\"'").upper()


malos = {k: v for k, v in EQUIVALENCIAS.items() if v not in CODIGOS_EXPERTA}
if malos:
    print("Ojo: estos destinos no son códigos de la experta:", malos)
equiv = {normalizar(k): v for k, v in EQUIVALENCIAS.items()}
for c in CODIGOS_EXPERTA:                 # si ya usó los códigos de la experta, se respetan
    equiv.setdefault(normalizar(c), c)
mias["codigo_experta"] = mias["categoria"].map(lambda c: equiv.get(normalizar(c), "sin_equivalencia"))

sin = mias.loc[mias.codigo_experta == "sin_equivalencia", "categoria"].value_counts()
if len(sin):
    print("Estas categorías suyas no tienen equivalencia y van a contar como desacuerdo:")
    print(sin.to_string())
else:
    print("Todas sus categorías tienen equivalencia.")

## Paso 5 · El acuerdo con la experta

El **porcentaje de acuerdo** dice en cuántas llamadas coinciden. El **kappa de Cohen** descuenta el acuerdo que se daría por azar: 1 es acuerdo perfecto y 0 es lo que se esperaría adivinando. Se comparan solo las llamadas que están en la muestra de la experta (hasta 60). Después lea los desacuerdos: ¿quién tiene razón, su IA o la experta? ¿O le falta una regla a su libro?

In [ ]:
from sklearn.metrics import cohen_kappa_score


def coma(x, d=1):
    return f"{x:.{d}f}".replace(".", ",")


comp = experta.merge(mias, on="id_llamada", how="inner")
comp = comp[comp.categoria != "ERROR"]
if comp.empty:
    raise ValueError("No hay llamadas en común con la muestra de la experta: revise que codificó las 80.")
acuerdo = (comp.codigo_experta == comp.repertorio).mean()
kappa = cohen_kappa_score(comp.repertorio, comp.codigo_experta)
print(f"Llamadas comparadas: {len(comp)} de las {len(experta)} de la muestra experta")
print(f"Acuerdo: {coma(100 * acuerdo)} % · Kappa de Cohen: {coma(kappa, 2)}\n")
display(pd.crosstab(comp.repertorio, comp.codigo_experta, rownames=["Experta"], colnames=["Usted"]))

desacuerdos = comp[comp.codigo_experta != comp.repertorio]
print(f"\n{len(desacuerdos)} desacuerdos\n")
for f in desacuerdos.itertuples():
    dificil = " · la experta la marcó como caso que no encaja" if f.ambigua == 1 else ""
    print(f"{f.id_llamada} · experta: {f.repertorio} · usted: {f.categoria} → {f.codigo_experta}{dificil}")
    print(f"   {f.resumen}")
    if f.justificacion:
        print(f"   Su IA se apoyó en: «{f.justificacion}»")
    print()

## Paso 6 · Los casos que no encajan

Los casos que no encajan son el material más valioso: obligan a afinar el libro de códigos. Es el paso de *refinamiento* de Nelson. La celda de abajo le pide a la IA las cinco llamadas que peor encajan en **su** libro y una regla para resolver cada una. Léalas y decida si la regla propuesta tiene sentido.

In [ ]:
import re

try:
    from google.colab import ai
except ImportError:
    ai = None

with open("mi_libro_de_codigos.md", encoding="utf-8") as fh:
    libro = fh.read()
ids_dudosos = set(mias.loc[mias.encaja.astype(str).str.lower().str.startswith("n"), "id_llamada"])
if "desacuerdos" in globals():
    ids_dudosos |= set(desacuerdos.id_llamada)
candidatas = llamadas[llamadas.id_llamada.isin(ids_dudosos)]
if len(candidatas) < 5:                      # si casi todo encajó, se le muestran todas
    candidatas = llamadas
lista = "\n".join(f"{f.id_llamada}: {f.resumen}" for f in candidatas.head(40).itertuples())

PROMPT_CINCO = f"""Eres un asistente de investigación cualitativa. Abajo están mi libro de códigos y una lista de llamadas al 123 que no encajaron bien o en las que mi codificación no coincidió con la de una experta.
Escoge las 5 llamadas que peor encajan en mi libro de códigos. Para cada una escribe: el id, en una frase por qué no encaja y qué regla o categoría nueva le agregarías al libro para resolverla. Usa solo llamadas de la lista.

LIBRO DE CÓDIGOS:
{libro}

LLAMADAS:
{lista}"""

if ai is not None:
    respuesta = ai.generate_text(PROMPT_CINCO)
    print(respuesta)
    print("\n— Las cinco llamadas, completas —\n")
    ids = list(dict.fromkeys(re.findall(r"\bL\d{3}\b", respuesta)))[:5]
    for f in llamadas[llamadas.id_llamada.isin(ids)].itertuples():
        print(f"{f.id_llamada}: {f.resumen}\n")
else:
    print("Copie este prompt en gemini.google.com:\n")
    print(PROMPT_CINCO)

## Para su bitácora

```text
Estación 2 · Mis categorías: __ · Acuerdo con la experta: __ % (kappa __) · Lo que mi libro no distinguía: __ · La regla que le agregaría: __ · Lo que hizo la IA y lo que puse yo: __
```

## Al final: el libro de códigos de la experta

*Léalo después de comparar, no antes. Fíjese en la regla de desempate: es lo que más le cuesta a una IA y lo que más le aporta el criterio cualitativo.*

---

### Libro de códigos · Repertorios de violencia en llamadas al 123

**Ciudad del Valle · versión 1 · octubre de 2026**
*Material de clase. La ciudad, los barrios y las llamadas son simulados.*

#### Para qué sirve

Este libro convierte la lectura experta de una llamada en un código que cualquier persona, o una IA, pueda aplicar igual. Las categorías no salieron de un algoritmo: salieron de leer cientos de minutas y preguntarse **qué forma toma la violencia** en cada una. Eso es un *repertorio*: la manera reconocible en que ocurre un conflicto (en la casa o en la calle, con golpes o con palabras, para quitar algo o para intimidar).

#### Reglas generales

1. **Unidad de codificación:** la llamada. Cada llamada recibe **un solo código**.
2. **Se codifica lo que dice la minuta**, no lo que uno supone. Si la minuta no dice que son pareja, no se asume que lo son.
3. **Se codifica el hecho que motiva la llamada.** Los antecedentes («la semana pasada lo empujó») ayudan a entender, pero no cambian el código del hecho de hoy.
4. Si la llamada mezcla repertorios, se aplica la **regla de desempate** del final.
5. Si después de aplicarla todavía duda, ponga el código que resulte y marque `duda = 1`. Esas llamadas son las más valiosas: son *los casos que no encajan*.

**Códigos válidos:** `vif_pareja`, `rina`, `amenaza`, `hurto_violento`, `acoso_mujer`, `convivencia`, `otro`.

---

#### 1 · `vif_pareja` · Violencia intrafamiliar o de pareja

**Definición.** Agresión física, verbal, psicológica o económica entre personas con vínculo de pareja (actual o pasada) o de familia, sin importar dónde ocurra.

**Incluye.**
- Golpes, empujones, insultos, gritos humillantes, encierro, daño de pertenencias o control del dinero entre pareja, expareja o familiares.
- Amenazas de una expareja, aunque ocurran en la calle o por mensajes.
- Agresiones de hijos, nietos, yernos o padrastros contra otros miembros del hogar.
- Llamadas de terceros (vecinos, celadores) **cuando la minuta establece el vínculo** («cree que es la pareja que vive allí»).

**Excluye.**
- Discusiones con gritos donde no se sabe quiénes son ni si hay agresión → `convivencia`.
- Conflictos entre personas que viven juntas **sin ser familia** (arrendador y arrendatario, compañeros de apartamento) → según el hecho: `amenaza`, `rina` o `convivencia`.
- Amenazas de la expareja **de otra persona** (por ejemplo, la exnovia del compañero) → `amenaza`: el vínculo que cuenta es el de agresor y víctima.

**Ejemplo.** «Señora reporta que su compañero la empujó contra la nevera durante una discusión en la cocina; él sigue en la casa. Pide que llegue una patrulla.»

#### 2 · `rina` · Riña en espacio público

**Definición.** Agresión física entre personas **sin** vínculo de pareja o familia, en la calle, un parque, una cancha, una vía o un establecimiento abierto al público.

**Incluye.**
- Peleas a golpes entre clientes de un bar, hinchas, jugadores, conductores después de un choque o vecinos en la calle.
- Enfrentamientos entre grupos, con o sin objetos lanzados.
- Agresiones físicas a trabajadores de un establecimiento (meseros, administradores) que no buscan quitarles un bien.

**Excluye.**
- Pelea entre familiares o pareja, aunque sea en la calle → `vif_pareja`.
- Pelea que empieza por un robo o un intento de robo → `hurto_violento`.
- Pelea que empieza porque alguien acosa a una mujer → `acoso_mujer`.
- Discusión sin contacto físico → `amenaza` (si hay intimidación) o `convivencia` (si no la hay).

**Ejemplo.** «Vigilante reporta que cuatro hombres se están golpeando frente a una licorera; hay botellas rotas en el andén. Solicita presencia policial.»

#### 3 · `amenaza` · Amenaza o intimidación sin agresión física

**Definición.** Anuncio de un daño futuro, o conducta para atemorizar, **sin contacto físico** en el hecho que motiva la llamada y sin vínculo de pareja o familia entre agresor y víctima.

**Incluye.**
- Amenazas verbales o por mensajes de vecinos, arrendadores, clientes, exsocios, compañeros de trabajo o desconocidos.
- Exigencias de dinero **a futuro** («una cuota semanal para cuidarle el negocio»), aunque no se lleven nada.
- Gestos intimidantes, seguimientos o vigilancia de la casa por personas sin vínculo.

**Excluye.**
- Si hubo contacto físico hoy → `rina` (o el repertorio que corresponda).
- Si la amenaza es para quitar un bien **en ese momento** («deme el celular») → `hurto_violento`.
- Si quien amenaza es pareja, expareja o familiar → `vif_pareja`.
- Si la intimidación se dirige a una mujer en espacio público, con acoso o persecución → `acoso_mujer`.
- Insultos o groserías sin anuncio de daño → `convivencia`.

**Ejemplo.** «Ciudadano reporta que el vecino del lote de al lado le dijo que si vuelve a abrir la reja le va a hacer daño; no hubo contacto. Pide orientación.»

#### 4 · `hurto_violento` · Hurto con violencia o intimidación

**Definición.** Quitarle, o intentar quitarle, a una persona un bien que lleva consigo o tiene a su cargo, **para quedarse con él**, usando fuerza o intimidación.

**Incluye.**
- Raponazos (arrebatar el celular, el bolso o la cadena), atracos a personas, a locales, a taxistas o a domiciliarios.
- Intentos de robo con intimidación aunque no se lleven nada («le exigió el celular diciéndole que iba armado; salió corriendo»).
- Robos que terminan en pelea entre la víctima, los vecinos y el agresor.

**Excluye.**
- Robo entre familiares (el hijo que se lleva el televisor de la casa) → `vif_pareja`.
- Exigencias de dinero a futuro (extorsión) → `amenaza`.
- Quitar un objeto para obligar a alguien a algo y devolverlo (no hay ánimo de quedarse con él) → el repertorio de la conducta, casi siempre `acoso_mujer`.
- Hurtos sin violencia ni intimidación (se dio cuenta después de que le faltaba la billetera) → `otro`.

**Ejemplo.** «Estudiante reporta que un hombre le dijo que estaba armado y le exigió el celular en el paradero; se lo entregó y el hombre huyó a pie. Solicita patrulla.»

#### 5 · `acoso_mujer` · Acoso o violencia contra la mujer en espacio público

**Definición.** Conducta dirigida a una mujer **por ser mujer**, en la calle, el transporte o un lugar abierto al público, por parte de alguien que no es su pareja ni su familiar: comentarios sobre su cuerpo, persecución, bloqueo del paso, contacto no consentido, grabación sin permiso o agresión cuando ella rechaza el acercamiento.

**Incluye.**
- Seguimientos, piropos ofensivos, insistencia para que dé su número o se suba a un carro.
- Contacto no consentido en el transporte o en la vía.
- Agresión a una mujer porque rechazó el acercamiento, y peleas que se desatan cuando alguien la defiende del acosador.

**Excluye.**
- Si el agresor es pareja, expareja o familiar → `vif_pareja`.
- Si el propósito es quitarle un bien → `hurto_violento`.
- Amenazas a una mujer por una deuda, un negocio o un conflicto vecinal, sin componente de acoso → `amenaza`.

**Ejemplo.** «Joven reporta que un hombre se sentó a su lado en una banca del parque, le hace comentarios sobre su cuerpo y no se aleja aunque ella se cambió de puesto. Pide que llegue una patrulla.»

#### 6 · `convivencia` · Convivencia sin violencia

**Definición.** Conflictos de la vida en común sin agresión física, sin amenaza y sin despojo: ruido, fiestas, animales, basuras, obras, parqueo, consumo en espacio público.

**Incluye.**
- Música a alto volumen, fiestas, obras de noche.
- Perros sueltos o que ladran, animales de cría en zona residencial.
- Escombros, quemas de basura, vehículos mal parqueados.
- Discusiones a gritos en las que no se sabe quiénes son ni si hay agresión.
- Insultos o groserías sin anuncio de daño.

**Excluye.**
- Si el conflicto escala a una amenaza → `amenaza`.
- Si escala a golpes → `rina` (o `vif_pareja`, si hay vínculo).
- Si la minuta dice que es una pareja que se insulta → `vif_pareja`.

**Ejemplo.** «Vecina reporta que en la casa de enseguida tienen un equipo de sonido a todo volumen desde el mediodía; no ha habido discusiones. Pide el llamado de atención.»

#### `otro` · No corresponde a ningún repertorio

**Definición.** La llamada describe otra cosa (accidente sin agresión, emergencia médica, persona extraviada, hurto sin violencia) o no trae información suficiente para saber qué pasó.

**Ejemplo.** «Ciudadano reporta un choque entre dos carros sin heridos; los conductores están conversando. Pide agente de tránsito.»

**Úselo poco.** Si la llamada describe un conflicto que se parece a varios repertorios, no es `otro`: es un caso para la regla de desempate.

---

#### Regla de desempate

Cuando una llamada mezcla repertorios, responda estas preguntas **en orden**. Gana la primera que se responda con **sí**.

| Orden | Pregunta | Si la respuesta es sí |
|---|---|---|
| 1 | ¿Agresor y víctima son pareja, expareja o familia? | `vif_pareja` |
| 2 | ¿Alguien quitó o intentó quitar un bien en ese momento, para quedarse con él, con fuerza o intimidación? | `hurto_violento` |
| 3 | ¿La conducta va contra una mujer en espacio público, por ser mujer, de parte de alguien que no es su pareja ni su familiar? | `acoso_mujer` |
| 4 | ¿Hubo agresión física hoy, en espacio público o en un establecimiento abierto al público? | `rina` |
| 5 | ¿Hubo amenaza o intimidación sin contacto físico? | `amenaza` |
| 6 | ¿Hay un conflicto de convivencia sin violencia? | `convivencia` |
| 7 | Ninguna de las anteriores | `otro` |

**Por qué este orden.** La relación entre las personas cambia el tipo de riesgo (quien agrede vuelve a la casa), por eso va primero. El despojo define el delito aunque termine en pelea. El acoso a una mujer es el origen de la pelea que viene después, no al revés. Y lo físico pesa más que lo verbal, y lo verbal más que lo que solo incomoda.

**Tres casos que suelen confundir.**
- *Una pareja pelea en la calle y otros se meten:* `vif_pareja` (pregunta 1), aunque haya riña.
- *Discusión por el ruido que termina en «lo voy a sacar a golpes»:* `amenaza` (pregunta 5), no `convivencia`.
- *Le exigieron el celular, pero salió corriendo:* `hurto_violento` (pregunta 2): el intento cuenta.
